In [ ]:
# 6.1: Mapa Interativo de Sinistros de Trânsito em Matão/SP
import pandas as pd
import plotly.express as px
import os

# Carregar dados limpos
df = pd.read_csv('../data/matao_limpo.csv', parse_dates=['data'])

# 1. Filtrar registros com coordenadas válidas dentro da região de Matão
df_mapa = df.dropna(subset=['latitude', 'longitude']).copy()

# Garantir limites geográficos de Matão e acessos rodoviários
filtro_matao = (
    (df_mapa['latitude'].between(-21.80, -21.40)) & 
    (df_mapa['longitude'].between(-48.60, -48.10))
)
df_mapa = df_mapa[filtro_matao].copy()

# Formatar data para exibição no tooltip
df_mapa['data_texto'] = df_mapa['data'].dt.strftime('%d/%m/%Y')

# 2. Definir ordem e paleta semântica de gravidade
ordem_gravidade = ['Com óbito', 'Com feridos graves', 'Com feridos leves', 'Sem vítimas']
cores_gravidade = {
    'Com óbito': '#d90429',        # Vermelho vivo
    'Com feridos graves': '#f77f00', # Laranja
    'Com feridos leves': '#e5b124',  # Dourado/Amarelo escuro
    'Sem vítimas': '#2b5c8f'         # Azul
}

# 3. Construir o Mapa Interativo com Plotly
fig = px.scatter_map(
    df_mapa,
    lat='latitude',
    lon='longitude',
    color='gravidade',
    category_orders={'gravidade': ordem_gravidade},
    color_discrete_map=cores_gravidade,
    hover_name='tipo_sinistro',
    hover_data={
        'latitude': False,
        'longitude': False,
        'gravidade': True,
        'data_texto': True,
        'hora': True,
        'tipo_via': True,
        'logradouro': True,
        'total_vitimas': True
    },
    zoom=12.2,
    center={"lat": -21.6035, "lon": -48.3645},
    map_style='open-street-map',
    height=680,
    title='Mapeamento dos Sinistros de Trânsito em Matão/SP (2015-2026)'
)

fig.update_layout(
    margin=dict(l=0, r=0, t=40, b=0),
    legend=dict(
        title=dict(text='<b>Gravidade</b>'),
        bgcolor='rgba(255, 255, 255, 0.85)',
        bordercolor='#cccccc',
        borderwidth=1,
        x=0.01,
        y=0.99
    )
)

# 4. Salvar como página HTML interativa independente
caminho_html = '../output/graficos/mapa_acidentes.html'
fig.write_html(caminho_html)

print("--- Mapa Gerado com Sucesso ---")
print(f"Total de sinistros mapeados: {len(df_mapa)}")
print(f"Arquivo interativo salvo em: {caminho_html}")

# Exibir no próprio Jupyter Notebook
fig.show()
